In [0]:
# =============================================================================
# 14_ingest_thresholds.py
# =============================================================================
# CADENCE:   One-off / manual re-run when source CSV changes
# PURPOSE:   Load flood threshold definitions from the restricted volume into
#            Delta, making them queryable alongside the rest of the pipeline.
# SOURCE:    /Volumes/prd_dash_lab/flood_forecasting_restricted/<your_file>.csv
# OUTPUT:    prd_dash_lab.flood_forecasting_restricted.thresholds_all
#
# NOTES:
#   - Raw source is preserved in full; deduplication happens downstream.
#   - Schema is inferred from the CSV on first load; overwriteSchema is enabled
#     so re-runs survive column order changes in the source file.
# =============================================================================

import os
import pandas as pd
from dotenv import load_dotenv
from pyspark.dbutils import DBUtils

import config

load_dotenv()
dbutils = DBUtils(spark)

RESTRICTED_CATALOG = "prd_dash_lab"
RESTRICTED_SCHEMA  = "flood_forecasting_restricted"
RESTRICTED_VOLUME  = "shared_external_volume"
CSV_PATH           = f"/Volumes/{RESTRICTED_CATALOG}/{RESTRICTED_SCHEMA}/{RESTRICTED_VOLUME}/thresholds_Feb.csv"
OUTPUT_TABLE       = f"{RESTRICTED_CATALOG}.{RESTRICTED_SCHEMA}.ea_thresholds_all"


In [0]:
# =============================================================================
# 1. Load source CSV
# =============================================================================

# Read as pandas first -- the CSV lives in a Volume, not a Hive path,
# so Spark cannot read it directly via spark.read.csv.
pdf = pd.read_csv(CSV_PATH, encoding="latin-1")

print(f"Source rows loaded: {len(pdf)}")
print(f"Columns: {pdf.columns.tolist()}")

In [0]:
# =============================================================================
# 2. Convert to Spark and write to Delta
# =============================================================================

# Create without explicit schema then let Delta infer types.
# This is safer than passing a schema when the source is externally managed
# and column types may shift between CSV exports.
df = spark.createDataFrame(pdf)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

row_count = spark.table(OUTPUT_TABLE).count()
print(f"Written to {OUTPUT_TABLE}: {row_count} rows")


In [0]:
# =============================================================================
# 3. Verify
# =============================================================================

display(spark.table(OUTPUT_TABLE).limit(10))

dbutils.notebook.exit("success")